# Model 3 — Experiment Artifact Grouped External Validation

This notebook validates the **selected controlled-tuning experiment artifact** against complete CIC-IDS2017 CSV files that were not used as its training data.

**Important:** this notebook does not retrain or modify the experiment artifact and does not overwrite production. It is a quality gate before promotion.

In [1]:
from pathlib import Path
import json, warnings, time
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT.name!="EVIDENT" and PROJECT_ROOT.parent!=PROJECT_ROOT:
    PROJECT_ROOT=PROJECT_ROOT.parent

DATA_DIR=PROJECT_ROOT/"data"/"raw"/"behaviour"
MODEL_DIR=PROJECT_ROOT/"models"/"behavior"
META_DIR=MODEL_DIR/"metadata"
EXP_PATH=MODEL_DIR/"experiments"/"network_threat_model_experiment.pkl"
OUT_PATH=META_DIR/"experiment_grouped_external_validation.json"
META_DIR.mkdir(parents=True,exist_ok=True)

assert EXP_PATH.is_file(), f"Experiment artifact not found: {EXP_PATH}"
artifact=joblib.load(EXP_PATH)
assert artifact.get("artifact_type")=="EVIDENT_MODEL3_EXPERIMENT"
features=artifact["feature_columns"]
binary_model=artifact["binary_model"]
family_model=artifact["family_model"]
print("Experiment:", EXP_PATH)
print("Config:", artifact["config"])
print("Feature count:", len(features))


Experiment: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_experiment.pkl
Config: {'name': 'higher_capacity', 'max_iter': 350, 'learning_rate': 0.06, 'max_leaf_nodes': 63, 'l2': 1.0}
Feature count: 67


In [2]:
DROP={
    "Flow ID","Source IP","Src IP","Destination IP","Dst IP",
    "Timestamp","Source Port","Src Port","Destination Port","Dst Port","Protocol"
}
CHUNK_SIZE=100000
CAP_PER_LABEL=50000

def clean_col(c): return " ".join(str(c).strip().replace("\ufeff","").split())
def norm(x): return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")
def family(x):
    x=norm(x)
    if x=="BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

def load_group(path):
    parts=[]; counts={}
    for chunk in pd.read_csv(path,chunksize=CHUNK_SIZE,low_memory=False):
        chunk.columns=[clean_col(c) for c in chunk.columns]
        if "Label" not in chunk.columns: continue
        chunk["Label"]=chunk["Label"].map(norm)
        selected=[]
        for label,part in chunk.groupby("Label",sort=False):
            used=counts.get(label,0); remaining=CAP_PER_LABEL-used
            if remaining<=0: continue
            selected.append(part.iloc[:min(remaining,len(part))])
            counts[label]=used+min(remaining,len(part))
        if selected: parts.append(pd.concat(selected,ignore_index=True))
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

def prepare(df):
    y=df["Label"].map(norm)
    fam=y.map(family)
    X=df.drop(columns=[c for c in DROP if c in df.columns]+["Label"],errors="ignore").copy()
    for c in X.columns: X[c]=pd.to_numeric(X[c],errors="coerce")
    X=X.replace([np.inf,-np.inf],np.nan)
    X=X.reindex(columns=features)
    return X, y, fam


## 1. Select representative holdout groups

The notebook prefers representative files for different attack families. It does not require hard-coded filenames: it scans available CIC-IDS2017 CSVs and chooses matches by filename. If fewer than three representative groups exist, run the notebook after adding the remaining dataset files.

In [4]:
all_files = sorted(DATA_DIR.glob("*.csv"))

print("Available CSV groups:")
for p in all_files:
    print(" -", p.name)

# For grouped external validation, evaluate every available CIC-IDS2017
# CSV group. Attack families are determined from the Label column,
# not from the filename.
selected = all_files.copy()

print("\nSelected holdouts:")
for p in selected:
    print(" -", p.name)

assert len(selected) >= 3, (
    "Need at least 3 CSV groups for grouped external validation."
)

print(f"\nTotal holdout groups: {len(selected)}")

Available CSV groups:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

Selected holdouts:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv

Total holdout groups: 8


In [5]:
def evaluate_file(path):
    df=load_group(path)
    X,y,fam=prepare(df)
    start=time.perf_counter()
    pred= binary_model.predict(X)
    elapsed=(time.perf_counter()-start)*1000/max(len(X),1)
    bf=precision_recall_fscore_support(y.map(lambda x: 0 if x=="BENIGN" else 1),pred,average="binary",zero_division=0)[2]
    attack=fam!="BENIGN"
    if attack.any():
        true=fam[attack]
        fp=family_model.predict(X.loc[attack])
        ff=precision_recall_fscore_support(true,fp,average="macro",zero_division=0)[2]
        fam_acc=accuracy_score(true,fp)
        fam_report=classification_report(true,fp,output_dict=True,zero_division=0)
    else:
        ff=None; fam_acc=None; fam_report=None
    return {
        "file":path.name,"rows":int(len(X)),
        "original_labels":sorted(y.unique().tolist()),
        "attack_families":sorted(fam[attack].unique().tolist()),
        "binary_f1":float(bf),
        "family_macro_f1":None if ff is None else float(ff),
        "family_accuracy":None if fam_acc is None else float(fam_acc),
        "binary_latency_ms_per_row":float(elapsed),
        "binary_confusion_matrix":confusion_matrix(y.map(lambda x: 0 if x=="BENIGN" else 1),pred).tolist(),
        "family_classification_report":fam_report
    }

results=[]
for p in selected:
    print("\nEvaluating:",p.name)
    r=evaluate_file(p); results.append(r)
    print({k:r[k] for k in ["file","rows","binary_f1","family_macro_f1","attack_families"]})



Evaluating: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
{'file': 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'rows': 100000, 'binary_f1': 0.9978744848369939, 'family_macro_f1': 1.0, 'attack_families': ['DOS_DDOS']}

Evaluating: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
{'file': 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'rows': 100000, 'binary_f1': 0.9940751933514921, 'family_macro_f1': 0.24999249977499324, 'attack_families': ['PORT_SCAN']}

Evaluating: Friday-WorkingHours-Morning.pcap_ISCX.csv
{'file': 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'rows': 51966, 'binary_f1': 0.9219047619047619, 'family_macro_f1': 0.49987280590180616, 'attack_families': ['BOTNET']}

Evaluating: Monday-WorkingHours.pcap_ISCX.csv
{'file': 'Monday-WorkingHours.pcap_ISCX.csv', 'rows': 50000, 'binary_f1': 0.0, 'family_macro_f1': None, 'attack_families': []}

Evaluating: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
{'file': 'Thursday-WorkingHours-Afternoon-

In [6]:
valid=[r for r in results if r["family_macro_f1"] is not None]
avg_binary=float(np.mean([r["binary_f1"] for r in results]))
avg_family=float(np.mean([r["family_macro_f1"] for r in valid])) if valid else None
min_binary=float(np.min([r["binary_f1"] for r in results]))
min_family=float(np.min([r["family_macro_f1"] for r in valid])) if valid else None

summary={
    "validation_type":"grouped_external_validation_of_experiment_artifact",
    "artifact":str(EXP_PATH),
    "artifact_config":artifact["config"],
    "holdout_count":len(results),
    "aggregate":{
        "average_binary_f1":avg_binary,
        "average_family_macro_f1":avg_family,
        "minimum_binary_f1":min_binary,
        "minimum_family_macro_f1":min_family
    },
    "holdouts":results,
    "promotion_status":"REVIEW_REQUIRED"
}
OUT_PATH.write_text(json.dumps(summary,indent=2),encoding="utf-8")
print("\n=== EXPERIMENT QUALITY GATE ===")
print(json.dumps(summary["aggregate"],indent=2))
print("\nSaved:",OUT_PATH)



=== EXPERIMENT QUALITY GATE ===
{
  "average_binary_f1": 0.7403055333061107,
  "average_family_macro_f1": 0.5646509317634137,
  "minimum_binary_f1": 0.0,
  "minimum_family_macro_f1": 0.24999249977499324
}

Saved: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\experiment_grouped_external_validation.json


## 2. Decision

Do not promote automatically from this notebook. Review the aggregate and per-file results. The important question is whether the higher-capacity candidate generalizes across multiple unseen file groups, especially beyond DDOS/PORTSCAN.

If the grouped results are acceptable, the next step is a controlled **promotion** of the experiment artifact into `models/behavior/production/`, followed by API/inference integration tests.